# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/FawadAhmad-bilal/flyrank-assignment-1/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

**The rule, in plain words:** A page deserves review if it gets real search visibility (impressions_90d >= 100 — no point flagging pages nobody sees), and either (a) it hasn't been updated in a long time, or (b) its click-through rate is falling short of what other pages at its own search-position tier typically earn. Both conditions get worse the longer/bigger the gap, so the score scales with the *size* of the problem, not just a yes/no flag.

**Reason codes the rule can output (one per row, priority order):**
- `stale_and_ctr_gap` — both problems present
- `ctr_underperforming` — only the CTR-vs-position gap
- `stale_visible` — only the staleness problem
- `no_flag` — visible, but neither problem shows up
- `low_visibility` — impressions_90d < 100, not enough traffic to judge anything reliably

**Leakage note:** `trend_direction` / `trend_pct` are the label source (data-dictionary rule #2) and are used ONLY below to check whether these signals are worth trusting — never as inputs to the score itself.

### Signal check 1 — Staleness (linked to the session's refresh flags)

In [1]:
import pandas as pd
pd.set_option("display.max_columns", 60)

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)  # EVAL ONLY, never a feature

review = df[df["impressions_90d"] >= 100].copy()  # visibility floor -- don't test staleness on pages nobody sees
tier_order = ["0-30", "31-90", "91-180", "181+"]
review["freshness_tier"] = pd.Categorical(review["freshness_tier"], categories=tier_order, ordered=True)

sig1 = review.groupby("freshness_tier", observed=True).agg(
    n=("content_id", "count"),
    decline_rate=("is_declining_label", "mean")
).reset_index()
print(sig1)

  freshness_tier      n  decline_rate
0           0-30  13735      0.582745
1          31-90    152      0.592105
2         91-180   8084      0.622464
3           181+     35      0.742857


**Verdict (Signal 1): CONFIRMED — but weak.** Decline rate rises with staleness across the reliable buckets: 58.3% (0-30d, n=13,735) -> 59.2% (31-90d, n=152) -> 62.2% (91-180d, n=8,084), against a 54.2% base rate. Direction is right, but the lift is modest — this signal alone barely separates pages. The 181+ bucket (n=35) is below the ~50-row floor from the signal-audit skill, so no verdict is claimed for it specifically, even though its 74.3% rate looks dramatic.

### Signal check 2 — CTR vs. position (linked to the session's CTR-fix logic)

In [2]:
# avg_position == 0 means "no position data" per the data dictionary -- exclude those rows
review2 = df[(df["impressions_90d"] >= 100) & (df["avg_position"] > 0)].copy()

# Weighted CTR per position tier (weight by impressions, not the mean of per-row rates --
# averaging raw rates would mis-weight low-traffic pages, per the auditing-signals skill)
bench = review2.groupby("position_tier").apply(
    lambda g: g["clicks_90d"].sum() / g["impressions_90d"].sum() * 100
).to_dict()
print("Weighted CTR benchmark per position tier:", {k: round(v,3) for k,v in bench.items()})

review2["tier_bench"] = review2["position_tier"].map(bench)
review2["ctr_below_benchmark"] = (review2["ctr"] < review2["tier_bench"]).astype(int)

sig2 = review2.groupby("ctr_below_benchmark").agg(
    n=("content_id", "count"),
    decline_rate=("is_declining_label", "mean")
).reset_index()
print(sig2)

Weighted CTR benchmark per position tier: {'deep': 0.039, 'page_1': 0.35, 'page_3_5': 0.155, 'striking': 0.347, 'top_3': 0.487}
   ctr_below_benchmark      n  decline_rate
0                    0   6603      0.521733
1                    1  15403      0.630202


**Verdict (Signal 2): CONFIRMED.** Pages whose CTR sits below their own position tier's benchmark decline at 63.0% (n=15,403), versus 52.2% for pages meeting/beating their tier's benchmark (n=6,603) — both buckets comfortably clear the sample-size floor. This is a clearly stronger, more reliable signal than staleness alone, so it carries more weight in the rule below.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [4]:
def score_row(row, bench):
    if row["impressions_90d"] < 100:
        return pd.Series([0.0, "low_visibility", "no_action_low_visibility"])

    # Staleness component: scales 0-40 with how stale the page is, capped at 365 days
    stale_component = min(row["days_since_last_update"], 365) / 365 * 40

    # CTR-gap component: scales 0-60 with how far CTR sits below its position tier's benchmark
    ctr_gap_component = 0.0
    has_position = row["avg_position"] > 0
    tier_bench = bench.get(row["position_tier"]) if has_position else None
    if tier_bench:
        gap = max(0.0, tier_bench - row["ctr"]) / tier_bench
        ctr_gap_component = min(gap, 1.0) * 60

    score = round(stale_component + ctr_gap_component, 1)

    stale_flag = row["days_since_last_update"] >= 90
    ctr_flag = ctr_gap_component > 0
    if stale_flag and ctr_flag:
        reason = "stale_and_ctr_gap"
    elif ctr_flag:
        reason = "ctr_underperforming"
    elif stale_flag:
        reason = "stale_visible"
    else:
        reason = "no_flag"

    if score >= 70:
        action = "refresh_urgent"
    elif score >= 35:
        action = "refresh_soon"
    else:
        action = "monitor"

    return pd.Series([score, reason, action])

df[["score", "reason_code", "action"]] = df.apply(lambda r: score_row(r, bench), axis=1)
ranked_queue = df.sort_values("score", ascending=False).reset_index(drop=True)

print(ranked_queue["action"].value_counts())
print()
print(ranked_queue["reason_code"].value_counts())

action
monitor                     10974
refresh_soon                 8934
no_action_low_visibility     7994
refresh_urgent               2098
Name: count, dtype: int64

reason_code
ctr_underperforming    9637
low_visibility         7994
stale_and_ctr_gap      5766
no_flag                4250
stale_visible          2353
Name: count, dtype: int64


In [5]:
import os
os.makedirs("../outputs", exist_ok=True)

output_cols = ["content_id", "client_id", "score", "reason_code", "action",
               "days_since_last_update", "avg_position", "ctr", "impressions_90d"]
ranked_queue[output_cols].to_csv("../outputs/baseline_action_score.csv", index=False)
print(f"Wrote {len(ranked_queue)} rows to work/outputs/baseline_action_score.csv")

Wrote 30000 rows to work/outputs/baseline_action_score.csv


**Precision@K check (building-baselines skill, step 4) — is this rule honestly beatable, not just lucky?**

In [6]:
base_rate = df["is_declining_label"].mean()
print(f"Base rate (no-model floor): {base_rate:.3f}")
for k in [20, 50, 100, 500]:
    p = ranked_queue.head(k)["is_declining_label"].mean()
    print(f"precision@{k}: {p:.3f}")

Base rate (no-model floor): 0.542
precision@20: 0.750
precision@50: 0.800
precision@100: 0.780
precision@500: 0.680


precision@20 = 0.750, precision@50 = 0.800 — both well above the 0.542 base rate, so the rule is doing real work, not coasting on the majority class. It should still be honestly beatable by a real model that can weigh more than two signals at once.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [7]:
top20 = ranked_queue.head(20)
cols = ["content_id", "score", "reason_code", "action", "days_since_last_update",
        "avg_position", "ctr", "impressions_90d"]
print(top20[cols].to_string(index=False))

          content_id  score       reason_code         action  days_since_last_update  avg_position  ctr  impressions_90d
content_02b0d6e30129   94.3 stale_and_ctr_gap refresh_urgent                     313           6.9 0.00              176
content_6476d1d8c050   94.3 stale_and_ctr_gap refresh_urgent                     313          67.8 0.00              304
content_d25a099b3726   93.4 stale_and_ctr_gap refresh_urgent                     305          64.5 0.00              202
content_f488400fca67   93.4 stale_and_ctr_gap refresh_urgent                     305           5.7 0.00              155
content_ab27c30d81f4   93.3 stale_and_ctr_gap refresh_urgent                     304           8.9 0.00              103
content_4f241bad48a3   85.9 stale_and_ctr_gap refresh_urgent                     236          19.1 0.00              285
content_b16bd7307b39   81.3 stale_and_ctr_gap refresh_urgent                     194          31.0 0.00             4590
content_ea41fe5cf292   80.1 stal

**Top-20 review** — one line each: action, why, and what would make it wrong.

1-5 (`content_02b0d6e30129`, `content_6476d1d8c050`, `content_d25a099b3726`, `content_f488400fca67`, `content_ab27c30d81f4`) — `refresh_urgent`, flagged for 300+ days stale with zero clicks despite 100-300 impressions. Wrong if: these are genuinely low-demand long-tail keywords where 0 clicks on ~150 impressions is normal, not a sign of decay — worth checking `search_volume` per row before trusting the flag.
6 (`content_4f241bad48a3`) — `refresh_urgent`, 236 days stale, zero clicks, HIGH competition. Wrong if: HIGH competition alone explains the zero CTR regardless of freshness, and refreshing won't move the position enough to matter.
7 (`content_b16bd7307b39`) — `refresh_urgent`, 194 days stale, 4,590 impressions, zero clicks. This is the highest-traffic zero-click page in the top 20 — wrong if there's a technical/indexing issue (e.g. wrong canonical, noindex) rather than a content-quality problem, since no amount of rewriting fixes a page that isn't being served in results the way GSC impressions suggest.
8-13 (the `content_type = informational`, `183-day-stale` cluster) — all `refresh_urgent` on the same reasoning (long stale + zero clicks). Wrong if: these all belong to one client whose CMS bulk-updated content on the same day 183 days ago, i.e. the staleness clock is a data artifact of one migration, not independent evidence per page.
14 (`content_5feee3994adb`) — `refresh_urgent`, 194 days stale, 7,812 impressions, 1 click (ctr=0.01). Wrong if: this is a low-intent informational query where even a well-optimized page would sit near zero CTR — the position (39.0) is deep enough that low CTR may be entirely explained by position, not content quality.
15-20 — same `stale_and_ctr_gap` pattern at shorter staleness (106-151 days). Wrong if: shorter staleness windows are less reliable signals than the 300+ day cases above — worth weighting these lower confidence than rows 1-5 even though the formula currently treats them on the same scale.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [9]:
# Weak-pick pattern check: how much of the top action tier is driven by ONE thing (zero CTR)?
refresh_urgent = ranked_queue[ranked_queue["action"] == "refresh_urgent"]
zero_ctr_share = (refresh_urgent["ctr"] == 0).mean()
print(f"Share of 'refresh_urgent' rows with exactly ctr=0: {zero_ctr_share:.3f}")
print(f"content_type diversity in refresh_urgent:\n{refresh_urgent['content_type'].value_counts()}")

Share of 'refresh_urgent' rows with exactly ctr=0: 0.999
content_type diversity in refresh_urgent:
content_type
keyword article    2055
feedly article       43
Name: count, dtype: int64


**Weak picks:** 99.9% of `refresh_urgent` rows have *exactly* zero clicks. That's not wrong — zero clicks on real impressions is genuine evidence — but it means the CTR-gap component saturates at its max (60) the moment CTR hits zero, so the 'urgent' tier can't currently tell apart a page with 0 clicks on 150 impressions from one with 0 clicks on 7,800 impressions (row 7 above). The rule also has zero `content_type` diversity at the top — every one of the top 20 is a `keyword article`. That could be a real pattern (keyword articles genuinely decay faster) or it could mean `feedly article` / `comparison article` pages have systematically different CTR benchmarks that this rule doesn't capture well. Worth checking before trusting the queue's composition at face value.

**Leakage check:**
- `trend_direction` / `trend_pct` — used ONLY in `is_declining_label` for evaluation (Signal checks and precision@K), never inside `score_row`.
- No `*_last_30d` / `*_prev_30d` columns were used as inputs — only `days_since_last_update`, `avg_position`, `ctr`, `position_tier`, and `impressions_90d`, all of which are knowable at the time a page would actually be reviewed, not derived from the future window.
- `client_id` / `content_id` are used for identification only, never as scoring inputs.

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*